# Day 15 — Capstone Review

This notebook verifies the full pipeline meets all requirements.
Everything is config-driven — no hardcoded paths.

In [1]:
# Parameters
CONFIG_PATH = "../config/config.yaml"

In [2]:
import sys, time, os
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

%load_ext autoreload
%autoreload 2

from pipeline import load_config, setup_logging
cfg    = load_config(CONFIG_PATH)
logger = setup_logging(cfg["paths"]["log_folder"])
logger.info("Day 15 capstone started")
print("Config and logging ready.")

2026-11-24 23:29:49 [INFO] Logging started — log file: ../logs\pipeline_20261124_232949.log


2026-11-24 23:29:49 [INFO] Day 15 capstone started


Config and logging ready.


## Req 2 — Read all Olist CSVs

In [3]:
from reader import read_csv

input_folder = cfg["paths"]["input_folder"]
csv_files    = sorted([f for f in os.listdir(input_folder) if f.endswith(".csv")])

print(f"Found {len(csv_files)} CSV files:")
for fname in csv_files:
    rows = read_csv(os.path.join(input_folder, fname))
    print(f"  {fname:45s} {len(rows):>5} rows")

Found 11 CSV files:
  bad_orders.csv                                    2 rows
  customers.csv                                   500 rows
  geolocation.csv                                2229 rows
  order_items.csv                                 576 rows
  order_payments.csv                              545 rows
  order_reviews.csv                               489 rows
  orders.csv                                      502 rows
  ordersdelimiter.csv                             502 rows
  product_category_name_translation.csv            71 rows
  products.csv                                    512 rows
  sellers.csv                                     317 rows


## Req 3 — Config-driven column mapping

In [4]:
from validator import load_column_mapping, apply_column_mapping

mapping = load_column_mapping(cfg["paths"]["column_mapping"])
print(f"Column mapping loaded from config ({len(mapping)} rules):")
for orig, standard in mapping.items():
    if orig != standard:
        print(f"  {orig:45s} -> {standard}")

Column mapping loaded from config (14 rules):
  order_purchase_timestamp                      -> purchased_at
  order_approved_at                             -> approved_at
  order_delivered_carrier_date                  -> carrier_delivered_at
  order_delivered_customer_date                 -> customer_delivered_at
  order_estimated_delivery_date                 -> estimated_delivery_at
  shipping_limit_date                           -> shipping_limit_at
  order_item_id                                 -> item_sequence


## Reqs 4 & 5 — Validation, bad_records, custom exceptions

In [5]:
from validator13 import validate_file
from schema import SchemaError, FileFormatError, SCHEMA

bad_path = cfg["paths"]["bad_records_file"]

for filename in cfg["files"]:
    filepath = os.path.join(input_folder, filename)
    required = cfg["pipeline"]["required_fields"].get(filename, [])
    try:
        s = validate_file(filepath, required, bad_records_path=bad_path,
                          rejection_threshold=cfg["thresholds"]["rejection_rate"])
        logger.info(f"{filename}: {s['valid']} valid, {s['rejected']} rejected")
        print(f"  OK  {filename:30s} valid={s['valid']}  rejected={s['rejected']}")
    except SchemaError as e:
        logger.error(str(e))
        print(f"  ERR {filename}: {e}")
    except FileFormatError as e:
        logger.warning(str(e))
        print(f"  SKIP {filename}: {e}")

2026-11-24 23:29:49 [INFO] orders.csv: 500 valid, 2 rejected


2026-11-24 23:29:49 [INFO] order_items.csv: 576 valid, 0 rejected


2026-11-24 23:29:49 [INFO] customers.csv: 500 valid, 0 rejected


  OK  orders.csv                     valid=500  rejected=2
  OK  order_items.csv                valid=576  rejected=0
  OK  customers.csv                  valid=500  rejected=0


## Reqs 6 & 7 — Logging to file + idempotent manifest

In [6]:
from pipeline import (
    load_manifest, save_manifest, is_already_processed,
    write_output_csv, write_output_json, write_run_summary,
)
from transformer import convert_types
from datetime import datetime

# Delete manifest to simulate a fresh run
manifest_path = cfg["paths"]["manifest_file"]
if Path(manifest_path).exists():
    Path(manifest_path).unlink()
    print("Manifest cleared for fresh run.")

start_time = time.time()
manifest   = load_manifest(manifest_path)
summaries  = []

for filename in cfg["files"]:
    if is_already_processed(filename, manifest):
        logger.info(f"SKIP {filename}")
        continue
    filepath = os.path.join(input_folder, filename)
    required = cfg["pipeline"]["required_fields"].get(filename, [])
    try:
        s = validate_file(filepath, required, bad_records_path=bad_path,
                          rejection_threshold=cfg["thresholds"]["rejection_rate"])
        rows = read_csv(filepath)
        rows = apply_column_mapping(rows, mapping)
        rows = convert_types(rows, cfg["pipeline"]["float_cols"],
                                   cfg["pipeline"]["datetime_cols"])
        stem = Path(filename).stem
        write_output_csv(rows,  os.path.join(cfg["paths"]["output_folder"], f"{stem}_clean.csv"))
        write_output_json(rows, os.path.join(cfg["paths"]["output_folder"], f"{stem}_clean.json"))
        manifest[filename] = datetime.now().isoformat()
        summaries.append(s)
        logger.info(f"Processed {filename}")
    except (SchemaError, FileFormatError) as e:
        logger.error(str(e))

save_manifest(manifest, manifest_path)
write_run_summary(summaries, start_time,
    os.path.join(cfg["paths"]["output_folder"], "run_summary.json"))

print(f"\nRun 1 complete — {len(summaries)} files processed.")
print("\nRun 2 (should skip all):")
manifest2 = load_manifest(manifest_path)
for f in cfg["files"]:
    status = "SKIP" if is_already_processed(f, manifest2) else "PROCESS"
    print(f"  {status}: {f}")

2026-11-24 23:29:49 [INFO] Processed orders.csv


2026-11-24 23:29:49 [INFO] Processed order_items.csv


2026-11-24 23:29:49 [INFO] Processed customers.csv


Manifest cleared for fresh run.

Run 1 complete — 3 files processed.

Run 2 (should skip all):
  SKIP: orders.csv
  SKIP: order_items.csv
  SKIP: customers.csv


## Req 8 — At least 5 passing unit tests

In [7]:
!python -m pytest ../tests/ -v --tb=short

============================= test session starts =============================
platform win32 -- Python 3.14.10, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\Arun\AppData\Local\Programs\Python\Python314\python.exe
cachedir: .pytest_cache
rootdir: D:\Arun\Tasks\python-git
collecting ... collected 19 items

..\tests\test_day12.py::test_read_csv_returns_list_of_dicts PASSED       [  5%]
..\tests\test_day12.py::test_read_csv_empty_file_returns_empty_list PASSED [ 10%]
..\tests\test_day12.py::test_to_snake_case_spaces PASSED                 [ 15%]
..\tests\test_day12.py::test_to_snake_case_camel PASSED                  [ 21%]
..\tests\test_day12.py::test_to_snake_case_already_snake PASSED          [ 26%]
..\tests\test_day12.py::test_to_snake_case_hyphens PASSED                [ 31%]
..\tests\test_day12.py::test_calculate_order_totals_basic PASSED         [ 36%]
..\tests\test_day12.py::test_calculate_order_totals_missing_freight PASSED [ 42%]
..\tests\test_day12.py::test_calculate_order_totals_em

## Req 10 — Live: add a file with an extra column via config

When the reviewer drops a new CSV (e.g. `new_file.csv` with extra columns):
1. Place the file in `data/`
2. Add its name to `files:` in `config/config.yaml`
3. Add required fields under `pipeline.required_fields` in `config/config.yaml`
4. Add any column renames to `config/column_mapping.json`
5. Re-run this notebook — the pipeline picks it up with no code changes

In [8]:
# Simulate reviewer adding a new file with an extra column
import csv

new_file = os.path.join(input_folder, "reviewer_file.csv")
with open(new_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["order_id", "customer_id", "extra_column"])
    writer.writerow(["a" * 32, "b" * 32, "some_extra_value"])
    writer.writerow(["c" * 32, "d" * 32, "another_value"])
print(f"Reviewer file created: {new_file}")

# Read and show it works without code changes — just config
rows = read_csv(new_file)
print(f"Columns: {list(rows[0].keys())}")
print(f"Rows: {len(rows)}")
print("Extra column value:", rows[0].get("extra_column"))
print("\nTo process this file through the full pipeline:")
print("  1. Add 'reviewer_file.csv' to files: in config.yaml")
print("  2. Add required fields under pipeline.required_fields")
print("  3. Re-run this notebook")

Reviewer file created: ../data\reviewer_file.csv
Columns: ['order_id', 'customer_id', 'extra_column']
Rows: 2
Extra column value: some_extra_value

To process this file through the full pipeline:
  1. Add 'reviewer_file.csv' to files: in config.yaml
  2. Add required fields under pipeline.required_fields
  3. Re-run this notebook
